In [1]:
from migration import ddl_resolver
from migration.ddl_resolver import DdlResolver
from src.utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer, CurDecomposerWriter
from migration.cur.metadata import CurMetadataProcessor
from migration.cur.generator import CurPySparkGenerator
from src.utils.source_rule_loader import load_all_source_rules

from src.paths import *

In [2]:
print(USERNAME)


output_root = PROJECT_ROOT / "output" / "migration"
all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()["cur"]

ext_giadung


In [3]:

# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_broker",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_address",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_crs",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_einvoice_customer_address",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment",


def get_paths(script_name):
    input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  f"{script_name}.sql"

    file_name = os.path.basename(input_file).replace('.sql', '')
    layer, sub_layer, source_name, base_table = parse_file_name(input_file)

    print(f"File gốc tại: {input_file}")

    return input_file, file_name, input_file

In [4]:
from jinja.environment import render_template


def run_migration_pipeline(script_name):
    input_file, file_name, input_file = get_paths(script_name)

    # ==========================================
    # BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
    # ==========================================
    decomposer = CurSqlDecomposer(source_rules)
    decomposed_script = decomposer.decompose(input_file)

    # Ghi file sub-SQL ra ổ đĩa
    writer = CurDecomposerWriter()
    writer.write(decomposed_script, output_root / file_name)
    print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")


    # ==========================================
    # BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
    # ==========================================
    processor = CurMetadataProcessor(source_rules)

    try:
        # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
        pipeline_config = processor.process(decomposed_script, input_file)

        # Ghi file YAML
        metadata_output_dir = output_root / file_name / "metadata"
        processor.write_yaml(pipeline_config, metadata_output_dir)

        print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
        print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
        print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
        print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

    except FileNotFoundError as e:
        print(f"❌ [Lỗi Bước 2]: {e}")
        print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


    # print("==========================================")
    # print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
    # print("==========================================")

    with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
        pipeline_config = yaml.safe_load(f)

    generator = CurPySparkGenerator(source_rules, output_mode="simple")
    ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)


    print("🎉 Hoàn tất toàn bộ Pipeline!")


In [5]:
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_log_field_value_change_dim_account.sql"
decomposer = CurSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

decomposed_script.target_table

processor = CurMetadataProcessor(source_rules)
pipeline_config = processor.process(decomposed_script, input_file)

# decomposer._extract_mutated_table_name(decomposed_script.source_blocks['DIM_ACCOUNT'].ast_nodes[0])


'ANALYZE TABLE ${cur_schema}.LOG_FIELD_VALUE_CHANGE PARTITION (SOURCE_KEY = 'DIM_ACCOUNT', YEAR_MONTH' contains unsupported syntax. Falling back to parsing as a 'Command'.
'ANALYZE TABLE ${cur_schema}.LOG_FIELD_VALUE_CHANGE PARTITION (SOURCE_KEY = 'DIM_ACCOUNT', YEAR_MONTH' contains unsupported syntax. Falling back to parsing as a 'Command'.


C:\Users\ext_giadung\projects\datalake-script\dml\cur\cur_log_field_value_change.sql


In [6]:

table_list = [
    # Từ ảnh 1
    # "log_field_value_change_dim_account",
    # "log_field_value_change_dim_account_bank",
    # "log_field_value_change_dim_account_cif",
    # "log_field_value_change_dim_address",
    # "log_field_value_change_dim_contact_email",
    # "log_field_value_change_dim_contact_mobile",
    # "log_field_value_change_dim_contact_office",
    # "fact_sbl_loan_position",
    # "fact_margin_position",
    # "fact_loan_payment_schedule_lms_loantranchechild2",
    # "exc_missing_trx_toms_count",
    # "exc_missing_trx_smf_count",
    # "exc_missing_trx_rak_count",
    # "exc_missing_trx_mhbos_t_rec_count",
    # "exc_missing_trx_mhbos_t_payt_count",
    # "exc_missing_trx_mhbos_t_ledger_count",
    # "exc_missing_trx_mhbos_t_glled_count",
    # "exc_missing_trx_mhbos_t_ctr_count",
    # "exc_missing_trx_mhbos_t_contract_count",
    # "exc_missing_trx_m21_o_count",
    # "exc_missing_trx_m21_a_count",
    # "exc_missing_trx_lms_count",
    # "exc_missing_acc_toms",
    # "exc_missing_acc_sbl",
    # "exc_missing_acc_rak",
    # "exc_missing_acc_mhbos",
    # "exc_missing_acc_m21_o",
    # "exc_missing_acc_m21_a",
    # "exc_missing_acc_lms",
    # "exc_customer_id_mapping",
    "exc_missing_trx_kdi_count",
    "exc_missing_trx_sbl_count",
    "exc_missing_trx_mhbos_t_setoff_count",

    # Từ ảnh 2
    # "exc_missing_acc_kdi",
    # "dim_sbl_loan",
    # "dim_joint_account_holder",
    # "dim_indicators_kdi",
    # "dim_indicators_lms",
    # "dim_indicators_m21",
    # "dim_indicators_mhbos",
    # "dim_indicators_rak",
    # "dim_indicators_sbl",
    # "dim_indicators_toms_ecorporate",
    # "dim_indicators_toms_eretail",
    # "dim_fund",
    # "dim_account_cif_rak_customer",
    # "dim_account_cif_sbl_client",
    # "dim_account_cif_snapshot",
    # "dim_account_cif_toms_eretail_cif",
    # "dim_account_cif_m21_customer",
    # "dim_account_cif_mhbos_m_client",

    # "ailab_mhbos_t_trust_acc",
    # "ailab_mhbos_t_ledger_hd",
    # "ailab_mhbos_t_ledger_dt",
    # "ailab_mhbos_t_contract",
    # "ailab_mhbos_m_client_ext",
    # "ailab_mhbos_m_client",
    # "ailab_mhbos_ecmit029r",
    # "ailab_mhbos_ecmit028r",
    # "ailab_dm_sbl_loan_position",
    # "ailab_dm_sbl_loan",
    # "ailab_dm_margin_position",
    # "ailab_dm_log_field_value_change",
    # "ailab_dm_digital_invest_position",
    # "ailab_dm_customer_custom",

    # Từ ảnh 3
    # "ailab_dm_customer",
    # "ailab_dm_cash_movement",
    # "ailab_dm_account_cif",

    # Từ ảnh 4
    # "einvoice_account"
]

for table in table_list:
    try:
        run_migration_pipeline(f"cur_{table}")
    except:
        run_migration_pipeline(f"cur_{table}")



File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\cur\cur_exc_missing_trx_kdi_count.sql
✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_exc_missing_trx_kdi_count\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: []
   -> File YAML đã lưu tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_exc_missing_trx_kdi_count\metadata\cur_exc_missing_trx_kdi_count.yaml
Processing source: UNKNOWN_SOURCE, from file: 88_unknown_blocks
File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\cur\cur_exc_missing_trx_kdi_count.sql
✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_exc_missing_trx_kdi_count\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: []
   -> File YAML đã

AttributeError: 'NoneType' object has no attribute 'db'

In [ ]:
from pathlib import Path

def merge_ddl_to_single_file(table_list: list, output_merged_path: str):
    """
    Reads DDL scripts for a list of tables and merges them into a single file.
    Ensures no table DDL is duplicated in the output.
    """
    output_file = Path(output_merged_path)
    # Ensure the parent directory for the merged file exists
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Base directory where individual DDL files are located
    base_ddl_dir = Path("C:/Users/ext_giadung/projects/hql_spark_bridge/output/migration/ddl/cur")

    # Use a set to keep track of processed tables and avoid duplicates
    processed_tables = set()

    # Open the merged file once in write mode to clear old content, then append
    with open(output_file, "w", encoding="utf-8") as merged_f:
        merged_f.write(f"-- ─────────────────────────────────────────────────────────\n")
        merged_f.write(f"-- MERGED DDL MIGRATION SCRIPT\n")
        merged_f.write(f"-- ─────────────────────────────────────────────────────────\n\n")

        for table in table_list:
            table_lower = table.lower().strip()

            # Skip if the table has already been processed in this run
            if table_lower in processed_tables:
                print(f"Table '{table_lower}' already processed. Skipping duplicate.")
                continue

            ddl_content = ""
            current_target_table = table_lower

            # 1. Try reading the existing DDL file first
            expected_path = base_ddl_dir / f"cur_{table_lower}.sql"

            try:
                ddl_content = expected_path.read_text(encoding="utf-8")
                print(f"Found existing DDL for: {table_lower}")

            except FileNotFoundError:
                # 2. Fallback: Decompose and generate DDL if file doesn't exist
                print(f"DDL file not found for '{table_lower}'. Triggering decomposer...")
                try:
                    # Fixed the double input_file assignment bug
                    input_file, file_name, _ = get_paths(f"cur_{table_lower}")

                    decomposer = CurSqlDecomposer(source_rules)
                    decomposed_script = decomposer.decompose(input_file)

                    current_target_table = decomposed_script.target_table.lower()

                    # Double-check if the decomposed target table was already processed
                    if current_target_table in processed_tables:
                        print(f"Decomposed target table '{current_target_table}' already processed. Skipping.")
                        continue

                    generated_path = base_ddl_dir / f"cur_{current_target_table}.sql"
                    ddl_content = generated_path.read_text(encoding="utf-8")

                except Exception as e:
                    print(f"[Error] Failed to decompose or read DDL for table {table_lower}: {e}")
                    continue

            # 3. Write to the merged file if valid content was retrieved
            if ddl_content.strip():
                merged_f.write(f"-- Block: cur_{current_target_table}\n")
                merged_f.write(ddl_content.strip())
                merged_f.write("\n\n-- ─────────────────────────────────────────────────────────\n\n")

                # Mark this table as completed to lock it from future duplicate writes
                processed_tables.add(current_target_table)
                if current_target_table != table_lower:
                    processed_tables.add(table_lower)

    print(f"\nSuccessfully merged {len(processed_tables)} unique table DDLs into: {output_file}")

In [ ]:
merge_ddl_to_single_file(table_list, r"C:\Users\ext_giadung\projects\hql_spark_bridge\docs\reference\testing\cur_ddl.sql")

In [ ]:
template_dir = PROJECT_ROOT / "template" / "migration"
model_folders = [d for d in template_dir.iterdir() if d.is_dir() and d.name.startswith('model_')]

for model_folder in model_folders:
    model_name = model_folder.name
    print(model_name.split("_")[-1].lower())
    # print(f"Processing model: {model_name}")
    # enricher = DdlResolver(source_rules=source_rules)
    # ddl_context = enricher.enrich(pipeline_config, model_type=model_name.split()[-1].lower())